# 04 `fpos` Model Progression

This notebook started as the main positive result of the thesis, but the **20-seed recording-disjoint sweep (`42` to `61`)** changes the tone. `fpos` still has several strong transfer recipes, yet the aggregate result is no longer “one universal best recipe”.

Unless otherwise noted, the benchmark tables below report those 20 seeds and the diagnostics pool the saved holdout predictions across the same splits.

**Questions answered here**
- Which `fpos` recipes actually survive the 20-seed sweep?
- Which split compositions and paired families favor which model?
- Which ablation blocks mattered most?
- Which structural differences across the paired datasets help explain those preferences?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("04_fpos_model_progression")
benchmark = build_benchmark_progress_table("fpos", scope="core")
ablation_winners = build_ablation_winner_table("fpos")
wave_ids = ["fpos_paired_context", "fpos_context_stack", "fpos_pre_waveform_unlabeled", "fpos_waveform_winner"]
family_pref_ids = ["fpos_paired_raw", *wave_ids]
seed_ids = ["fpos_paired_raw", *wave_ids]
waves = build_main_wave_summary("fpos", wave_ids)
seed_story = build_split_regime_bundle("fpos", seed_ids, split_profile_recipe_id="fpos_waveform_winner")
family_preference = build_fpos_lofo_main_family_preference_table()
family_context = build_family_structure_context_table("fpos", wave_ids)
winner_diag = build_prediction_diagnostics("fpos_waveform_winner", "fpos")
limitations = get_limitations_bundle()


## 1. Benchmark ladder

The main-text `fpos` ladder is intentionally compressed to the core comparison set: baseline floor, target-only reference, source-only transfer, context-first stacking, unlabeled structure, and waveform augmentation. It should be read as a methodological progression rather than as an exhaustive recipe catalog.


In [ ]:
display(benchmark[["label", "mae", "rmse", "r2", "bias", "calibration_slope", "delta_r2_vs_basic_transfer", "delta_mae_vs_basic_transfer", "notes"]])
save_table(benchmark, table_dir, "fpos_benchmark_ladder")
fig, _ = plot_benchmark_metric(benchmark, metric="r2", title="`fpos` benchmark ladder by R² (20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fpos_benchmark_r2")
fig


In [ ]:
fig, _ = plot_benchmark_metric(benchmark, metric="mae", title="`fpos` benchmark ladder by MAE (20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fpos_benchmark_mae")
fig


In [ ]:
milestone = pd.DataFrame([
    {"stage": "1  sanity floor", "model": "Dummy paired mean", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_dummy", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_dummy", "mae"].iloc[0])},
    {"stage": "2  paired baseline", "model": "Paired-only raw", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_paired_raw", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_paired_raw", "mae"].iloc[0])},
    {"stage": "3  source transfer", "model": "Hybrid-only XGBoost", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_hybrid_only", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_hybrid_only", "mae"].iloc[0])},
    {"stage": "4  paired context", "model": "Paired-only context", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_paired_context", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_paired_context", "mae"].iloc[0])},
    {"stage": "5  context stacking", "model": "Context-first source stack", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_context_stack", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_context_stack", "mae"].iloc[0])},
    {"stage": "6  + unlabeled paired", "model": "Pre-waveform unlabeled structure", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_pre_waveform_unlabeled", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_pre_waveform_unlabeled", "mae"].iloc[0])},
    {"stage": "7  + waveform embedding", "model": "Waveform-augmented structure", "r2": float(benchmark.loc[benchmark["recipe_id"] == "fpos_waveform_winner", "r2"].iloc[0]), "mae": float(benchmark.loc[benchmark["recipe_id"] == "fpos_waveform_winner", "mae"].iloc[0])},
])
display(milestone)
save_table(milestone, table_dir, "fpos_milestone_trajectory")


In [ ]:
show_saved_figure(
    fig_dir / "fpos_seed_stability.png",
    "These violin panels summarize the 20 recording-disjoint splits (`42` to `61`) for the retained compared `fpos` models after omitting the dummy and pure source-only baselines, so the thesis can separate one favorable split from the overall stability pattern.",
)
show_saved_figure(
    fig_dir / "fpos_fmiss_milestone_trajectory.png",
    "The milestone trajectory makes the asymmetry of the two targets visible: later transfer additions produce much larger gains for `fpos` than for `fmiss`.",
)


In [ ]:
top_mean_row = benchmark.sort_values(["r2", "mae"], ascending=[False, True]).iloc[0]
source_only_row = benchmark.loc[benchmark["recipe_id"] == "fpos_hybrid_only"].iloc[0]
best_rank_row = seed_story["summary"].iloc[0]
waveform_rank_row = seed_story["summary"].loc[seed_story["summary"]["recipe_id"] == "fpos_waveform_winner"].iloc[0]
display(Markdown(
    f"""
## Main 20-seed read

- The source-only transfer baseline reaches **R² {source_only_row['r2']:.4f}**.
- The best **mean R²** in the full ladder is **{top_mean_row['label']}** at **{top_mean_row['r2']:.4f}**.
- Across the competitive late-wave recipes, the best **mean seed rank** is **{best_rank_row['label']}** (mean rank **{best_rank_row['mean_rank']:.2f}**, **{int(best_rank_row['winner_count'])}** seed wins).
- The recipe named **`Waveform-augmented structure`** is still competitive (mean rank **{waveform_rank_row['mean_rank']:.2f}**, **{int(waveform_rank_row['winner_count'])}** seed wins), but the 20-seed sweep says the story is **regime-dependent rather than dominated by one universally best recipe**.
"""
))


## 2. What survived the 20-seed sweep?

A useful robustness question is not only “who has the highest average `R²`?”, but also “who keeps showing up near the top when the holdout recordings change?”


In [ ]:
display(seed_story["summary"])
save_table(seed_story["summary"], table_dir, "fpos_seed_rank_summary")


In [ ]:
import matplotlib.pyplot as plt

rank_pivot = seed_story["rank_pivot"].copy()
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(rank_pivot.to_numpy(dtype=float), cmap="YlGn_r", vmin=1, vmax=max(len(rank_pivot), 1), aspect="auto")
ax.grid(False)
ax.set_xticks(range(rank_pivot.shape[1]))
ax.set_xticklabels(rank_pivot.columns.astype(str).tolist(), rotation=0)
ax.set_yticks(range(rank_pivot.shape[0]))
ax.set_yticklabels(rank_pivot.index.tolist())
ax.set_xlabel("Recording-disjoint seed")
ax.set_title("Late-wave `fpos` rank by seed (20 recording-disjoint seeds, 42-61; 1 = best `R²`)", pad=10)
for row_idx, row_label in enumerate(rank_pivot.index):
    for col_idx, col_label in enumerate(rank_pivot.columns):
        value = rank_pivot.loc[row_label, col_label]
        if pd.notna(value):
            ax.text(col_idx, row_idx, int(value), ha="center", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Rank")
fig.tight_layout()
save_figure(fig, fig_dir, "fpos_seed_rank_heatmap")
fig


In [ ]:
display(seed_story["winner_table"][[
    "seed",
    "label",
    "r2",
    "mae",
    "true_mean",
    "true_std",
    "rows__PAIRED_BOYDEN",
    "rows__PAIRED_CRCNS_HC1",
    "rows__PAIRED_ENGLISH",
    "rows__PAIRED_KAMPFF",
    "rows__PAIRED_MEA64C_YGER",
]])
display(seed_story["winner_profile"])
save_table(seed_story["winner_table"], table_dir, "fpos_seed_winner_by_split")
save_table(seed_story["winner_profile"], table_dir, "fpos_seed_winner_profile")


In [ ]:
winner_profile = seed_story["winner_profile"].copy()
share_cols = [col for col in winner_profile.columns if col.startswith("share__")]
share_plot = winner_profile.loc[:, ["label", *share_cols]].copy()
share_plot = share_plot.rename(columns=lambda col: col.replace("share__PAIRED_", "") if col.startswith("share__") else col)
family_cols = [col for col in share_plot.columns if col != "label"]
fig, ax = plt.subplots(figsize=(8, 3.8))
left = np.zeros(len(share_plot), dtype=float)
palette = ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#AA3377"]
for color, col in zip(palette, family_cols):
    values = pd.to_numeric(share_plot[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    ax.barh(share_plot["label"], values, left=left, label=col, color=color, alpha=0.9)
    left = left + values
ax.set_xlim(0, 1)
ax.set_xlabel("Mean family share in seeds won by the model")
ax.set_title("Winning models prefer different holdout mixes", pad=10)
ax.legend(title="Held-out family", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
save_figure(fig, fig_dir, "fpos_split_winner_family_mix")
fig


---
### Thesis highlights — Split regime map and raw-vs-waveform complementarity

_Provenance._ These figures are thesis synthesis artifacts regenerated from frozen tables and aggregated seed metrics. They intentionally sit on top of the notebook analysis rather than replacing the exploratory cells above.

- **Split regime map:** Seeds are ordered by clustered family-share composition rather than by one hand-picked family. The second panel now shows the leading-recipe `R²` together with target difficulty on the same seed order.
- **Raw vs waveform complementarity:** This figure compares `Paired-only raw` against `Waveform-augmented structure` directly at the seed level. The claim is intentionally narrow: waveform gains are regime-dependent and modestly associated with slightly higher KAMPFF share, slightly lower ENGLISH share, and lower target `IQR` rather than with one absolute “waveform-friendly family”.


In [ ]:
show_saved_figure(
    ROOT / "figures" / "09_thesis_highlight_figures" / "split_regime_map.png",
    "This synthesis figure keeps the family-composition story in notebook 04, but the ordering is now driven by clustered split composition rather than a single family share.",
)
show_saved_figure(
    ROOT / "figures" / "09_thesis_highlight_figures" / "raw_vs_waveform_complementarity.png",
    "This is the thesis-ready comparison between the simple paired model and the waveform-enhanced stack; it explains complementarity rather than claiming a universal waveform advantage.",
)


## 3. Which ablation blocks mattered?

Each ablation group corresponds to one scientific question. The goal is not to show every run equally, but to make the causal story of the leading stack understandable.


In [ ]:
display(ablation_winners)
save_table(ablation_winners, table_dir, "fpos_ablation_winners")


In [ ]:
for group_name in ["source_supervision", "context", "unlabeled_paired_structure", "waveform_representation", "robustness"]:
    bundle = build_ablation_bundle("fpos", group_name, "recording_disjoint_main")
    display(Markdown(f"### {group_name.replace('_', ' ').title()}"))
    display(bundle["benchmark"][["label", "mae", "r2", "notes"]])
    save_table(bundle["benchmark"], table_dir, f"fpos_ablation_{group_name}")


## 4. Which families prefer which recipe?

The stricter family-held-out view is where the generalization story becomes interpretable. This comparison averages `R²` across `20` leave-one-family-out seeds and is kept aligned with the five main compared `fpos` models, excluding only the dummy and pure source-only baselines.


In [ ]:
display(family_preference["best"])
display(family_preference["r2_pivot"])
save_table(family_preference["best"], table_dir, "fpos_lofo_family_preference_best")
save_table(family_preference["r2_pivot"].reset_index(), table_dir, "fpos_lofo_family_preference_r2")


In [ ]:
import matplotlib.pyplot as plt

family_r2 = family_preference["r2_pivot"].copy()
fig, ax = plt.subplots(figsize=(8.4, 3.8))
im = ax.imshow(family_r2.to_numpy(dtype=float), cmap="RdYlGn", vmin=-0.25, vmax=0.85, aspect="auto")
ax.grid(False)
ax.set_xticks(range(family_r2.shape[1]))
ax.set_xticklabels(family_r2.columns.tolist(), rotation=24, ha="right")
ax.set_yticks(range(family_r2.shape[0]))
ax.set_yticklabels([name.replace("PAIRED_", "") for name in family_r2.index.tolist()])
ax.set_title("Mean held-out-family `R²` across the five main compared `fpos` models\n(20-seed leave-one-family-out aggregate)", pad=10)
for row_idx, family_name in enumerate(family_r2.index):
    for col_idx, label in enumerate(family_r2.columns):
        value = family_r2.loc[family_name, label]
        if pd.notna(value):
            ax.text(col_idx, row_idx, f"{value:.2f}", ha="center", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Mean R²")
fig.tight_layout()
save_figure(fig, fig_dir, "fpos_lofo_family_heatmap")
fig


In [ ]:
display(Markdown(
    """
**Online context note**

The columns `channel_context`, `duration_context`, and `source_context` below paraphrase the paired-study descriptions from **SpikeForest / eLife 2020 Table 2**. The remaining columns are computed from the local thesis package and the pooled 20-seed outputs.
"""
))
display(family_context[[
    "study_set",
    "source_context",
    "channel_context",
    "duration_context",
    "spikeforest_recordings",
    "packaged_recordings",
    "packaged_labeled_rows",
    "packaged_unlabeled_rows",
    "packaged_recording_fraction_vs_spikeforest",
    "target_mean",
    "target_std",
    "paired_pca_distance_from_hybrid",
    "best_recording_disjoint_label",
    "best_recording_disjoint_r2",
    "best_lofo_label",
    "best_lofo_r2",
]])
save_table(family_context, table_dir, "fpos_family_structure_context")


## 5. Family and recording behavior of the main waves

The thesis should not claim improvement unless it can show that gains are broad. These summaries make it clear whether a new wave improved many recordings or only a handful.


In [ ]:
display(waves["family"])
display(waves["recording"].head(20))
save_table(waves["family"], table_dir, "fpos_family_wave_summary")
save_table(waves["recording"], table_dir, "fpos_recording_wave_summary")
fig, _, family_pivot = plot_group_metric(waves["family"], group_col="study_set", metric="r2", title="`fpos` family R² by model wave")
save_figure(fig, fig_dir, "fpos_family_r2_by_wave")
save_table(family_pivot, table_dir, "fpos_family_r2_pivot")
fig


In [ ]:
recording_mean = (
    waves["recording"]
    .groupby(["label", "recording_key"], as_index=False)["r2"]
    .mean()
)
top_rec, bottom_rec = build_extreme_groups_table(recording_mean[recording_mean["label"] == WAVEFORM_AUGMENTED_LABEL], group_col="recording_key", metric="r2", top_n=12)
focus_recordings = pd.concat([top_rec, bottom_rec], ignore_index=True)["recording_key"].drop_duplicates().tolist()
recording_focus = waves["recording"][waves["recording"]["recording_key"].isin(focus_recordings)].copy()
display(top_rec)
display(bottom_rec)
save_table(top_rec, table_dir, "fpos_top_recordings")
save_table(bottom_rec, table_dir, "fpos_worst_recordings")
fig, _, rec_pivot = plot_group_metric(recording_focus, group_col="recording_key", metric="r2", title="`fpos` recording R² by model wave\n(top/bottom recordings from 20 recording-disjoint seeds)")
save_figure(fig, fig_dir, "fpos_top_recording_r2_by_wave")
save_table(rec_pivot, table_dir, "fpos_recording_r2_pivot")
fig


## 6. Split-specific robustness and diagnostics

The last question is whether the recording-disjoint favorite remains the strongest choice when the split gets harsher. The leave-one-family-out companion table shows that it does not.


In [ ]:
lofo_map = {
    "fpos_pre_waveform_unlabeled": "reference_anchor_stack_xgboost",
    "fpos_waveform_winner": "wf_embed_anchor_stack_xgboost",
}
protocol_compare = benchmark.loc[
    benchmark["recipe_id"].isin(["fpos_pre_waveform_unlabeled", "fpos_waveform_winner"]),
    ["recipe_id", "label", "r2", "mae"],
].rename(columns={"r2": "recording_disjoint_r2", "mae": "recording_disjoint_mae"})
lofo_companion = limitations.lofo_aggregate.copy()
lofo_companion["recipe_id"] = lofo_companion["variant_id"].map({value: key for key, value in lofo_map.items()})
protocol_compare = protocol_compare.merge(
    lofo_companion.loc[:, ["recipe_id", "macro_r2", "macro_mae", "weighted_r2", "weighted_mae", "worst_family_r2"]],
    on="recipe_id",
    how="left",
)
protocol_compare["macro_r2_gap"] = protocol_compare["macro_r2"] - protocol_compare["recording_disjoint_r2"]
protocol_compare["weighted_r2_gap"] = protocol_compare["weighted_r2"] - protocol_compare["recording_disjoint_r2"]
display(protocol_compare.sort_values("recording_disjoint_r2", ascending=False))
save_table(protocol_compare, table_dir, "fpos_protocol_regime_preference")

waveform_protocol = pd.DataFrame([
    {
        "model": WAVEFORM_AUGMENTED_LABEL,
        "protocol": "recording_disjoint_main",
        "r2": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "recording_disjoint_r2"].iloc[0]),
        "mae": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "recording_disjoint_mae"].iloc[0]),
        "delta_r2_vs_recording": 0.0,
        "delta_mae_vs_recording": 0.0,
    },
    {
        "model": WAVEFORM_AUGMENTED_LABEL,
        "protocol": "family_disjoint_lofo",
        "r2": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "macro_r2"].iloc[0]),
        "mae": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "macro_mae"].iloc[0]),
        "delta_r2_vs_recording": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "macro_r2_gap"].iloc[0]),
        "delta_mae_vs_recording": float(protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "macro_mae"].iloc[0] - protocol_compare.loc[protocol_compare["recipe_id"] == "fpos_waveform_winner", "recording_disjoint_mae"].iloc[0]),
    },
])
save_table(waveform_protocol, table_dir, "fpos_winner_protocol_comparison")


In [ ]:
import matplotlib.pyplot as plt

protocol_plot = protocol_compare.sort_values("recording_disjoint_r2", ascending=False).reset_index(drop=True)
x = np.arange(len(protocol_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(x - width / 2, protocol_plot["recording_disjoint_r2"], width=width, label="Recording-disjoint mean R²", color="#2E6DA4")
ax.bar(x + width / 2, protocol_plot["macro_r2"], width=width, label="Family-held-out macro R²", color="#CC6677")
ax.set_xticks(x)
ax.set_xticklabels(protocol_plot["label"], rotation=25, ha="right")
ax.set_ylabel("R²")
ax.set_title("Which `fpos` recipe is best depends on the split", pad=10)
ax.legend()
fig.tight_layout()
save_figure(fig, fig_dir, "fpos_protocol_regime_preference")
fig


In [ ]:
winner_predictions = winner_diag["predictions"]
display(winner_predictions.head(20))
fig, _ = plot_prediction_scatter(winner_predictions, title="Best `fpos` model: observed vs predicted")
save_figure(fig, fig_dir, "fpos_winner_prediction_scatter")
fig


In [ ]:
lofo_companion = limitations.lofo_aggregate[limitations.lofo_aggregate["variant_id"].isin(["reference_anchor_stack_xgboost", "wf_embed_anchor_stack_xgboost"])].copy()
display(lofo_companion)
save_table(lofo_companion, table_dir, "fpos_family_held_out_companion")


In [ ]:
best_rank_row = seed_story["summary"].iloc[0]
display(Markdown(
    f"""
## Key takeaways

- The 20-seed sweep does **not** support one universal `fpos` best recipe. The strongest late-wave recipe by mean seed rank is **{best_rank_row['label']}**, and several recipes lead on multiple individual seeds.
- Family structure explains part of that instability: **English dominates local coverage, CRCNS_HC1 is tiny and high-variance, and the packaged subset is an uneven slice of the larger SpikeForest study sets**.
- On the harsher family-held-out protocol, the retained comparison shifts slightly toward **`Waveform-augmented structure`**, even though **{best_rank_row['label']}** remains the stronger robust reference on the recording-disjoint benchmark.
- The waveform-augmented recipe still tracks the target reasonably on pooled holdout predictions (**MAE {winner_diag['mae']:.4f}**, correlation **{winner_diag['correlation']:.3f}**), but it should now be presented as a **regime-specific headline model**, not as an unconditional best recipe.
"""
))
